In [1]:
from platosim.simfile import SimFile
from platosim.simulation import Simulation
from platosim.validation import switchOffAllEffects
import os
import numpy as np
import matplotlib.pyplot as plt

In [2]:
sim = Simulation("Rebinning")
switchOffAllEffects(sim)
sim.outputDir = os.environ["PLATO_WORKDIR"]

# One full-frame exposure

sim["ObservingParameters/NumExposures"] = 1
sim["PSF/Model"] = "MappedGaussian"
dim = 4510#128*2#1024 
numSubPixels = sim["SubField/SubPixels"]
sim["SubField/NumRows"] = dim
sim["SubField/NumColumns"] = dim

sim["ControlHDF5Content/WriteSubPixelImages"] = "yes"

# Make sure no sources are located in the sub-field

# sim["ObservingParameters/DecPointing"] = -sim["ObservingParameters/DecPointing"]

print("Start")

output = sim.run(removeOutputFile = True)

image = output.getImage(0)
subPixelImage = output.getSubPixelImage(0)

print("Done")

Start


Done


In [3]:
def rebin(a, dim, numSubPixels):
    
    a_view = a.reshape(dim, numSubPixels, dim, numSubPixels)
    return a_view.sum(axis=3).sum(axis=1)

In [4]:
rebinned = rebin(subPixelImage, dim, numSubPixels)
print(np.sum(rebinned))
print(np.min(image), np.max(image))

120834840000.0
5835.0 637063600.0


In [5]:
print(np.abs(np.sum(image) - np.sum(subPixelImage)) / np.sum(image) * 100)

# print(np.abs(np.sum(image) - np.sum(rebinned)) / np.sum(image) * 100)

print(np.max(np.abs(rebinned - image) / image) * 100)

0.1259088981896639
4.510085318543133e-05
